<a href="https://colab.research.google.com/github/O-2wice/correctness-aware-nl-query-translation-ocel/blob/main/notebooks/02_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 Evaluation — Baselines (B1/B2/B3) + Constrained Pipeline (Method M)

**Part 1 — Baselines:** Implements and evaluates three unconstrained NL-to-SQL baselines.  
**Part 2 — Method M:** Artifact-review walkthrough of the constrained pipeline using saved results.

Comparison figures (ExecRate, DenAcc, latency, etc.) are all generated in `03_phase2_results.ipynb`.

**Outputs:**
- `outputs/reports/nb03_b1_results.csv`, `nb03_b2_results.csv`, `nb03_b3_results.csv`
- `outputs/reports/nb03_baseline_metrics.json`
- `outputs/reports/nb04_method_m_dev.csv`


## 0. Imports and Configuration

In [ ]:
import json, csv, hashlib, os, re, time, urllib.request, urllib.error
import pandas as pd
import numpy as np
import duckdb
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import sys

# Locate project root
_src = next((p / "src" for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists()), None)
if _src and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from nb_utils import (
    get_project_root, save_fig, progress,
    print_banner, print_rule, print_ok, print_warn, print_err, print_info,
    print_gate, print_metric_table, styled_df,
)

ROOT      = get_project_root()
OCEL_DIR  = ROOT / "data" / "processed" / "ocel"
CONFIGS   = ROOT / "configs"
REPORTS   = ROOT / "outputs" / "reports"
FIGURES   = ROOT / "outputs" / "figures"
BENCHMARK = ROOT / "benchmark" / "nl2ocel_benchmark_dev.csv"
for d in (REPORTS, FIGURES): d.mkdir(parents=True, exist_ok=True)

CATALOG   = CONFIGS / "schema_catalog.json"
WHITELIST = CONFIGS / "relation_whitelist.json"
CATALOG_PATH   = CATALOG
WHITELIST_PATH = WHITELIST

RUN_SUBSET = None  # set to list of QIDs to run a subset, None for all
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
MODEL      = os.environ.get("NL2OCEL_MODEL", "deepseek-chat")
TIMEOUT_S  = int(os.environ.get("NL2OCEL_TIMEOUT", "60"))
TEMPERATURE = float(os.environ.get("NL2OCEL_TEMPERATURE", "0.0"))
MAX_TOKENS  = int(os.environ.get("NL2OCEL_MAX_TOKENS", "1024"))
REPORTS_DIR = REPORTS
REPORT_FILES = {
    'b1': REPORTS / 'nb03_b1_results.csv',
    'b2': REPORTS / 'nb03_b2_results.csv',
    'b3': REPORTS / 'nb03_b3_results.csv',
    'mm': REPORTS / 'nb04_method_m_dev.csv',
}

# Backend selection: deepseek > ollama > artifact review
BACKEND = os.environ.get("NL2OCEL_BACKEND", "deepseek").lower()
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
LLM_OK = False
ARTIFACT_REVIEW = False

print_ok(f"ROOT = {ROOT}")


## 1. Backend Readiness Check


In [ ]:
print_banner('Section 1: Backend Readiness', 'Verify the selected backend or fall back to saved artifacts')

LLM_OK = False
ARTIFACT_REVIEW = False

if BACKEND == 'deepseek':
    if DEEPSEEK_API_KEY:
        LLM_OK = True
        print_ok('DeepSeek API key detected — ready to run reported baseline evaluation configuration.')
    else:
        print_warn('DEEPSEEK_API_KEY not set — cannot run fresh DeepSeek baseline calls in this notebook.')
elif BACKEND == 'ollama':
    try:
        req = urllib.request.Request(f'{OLLAMA_URL}/api/tags')
        with urllib.request.urlopen(req, timeout=5) as resp:
            tags_data = json.loads(resp.read())
        available_models = [m['name'] for m in tags_data.get('models', [])]
        print_ok(f'Ollama reachable. Available models: {available_models}')
        if any(MODEL in m for m in available_models):
            LLM_OK = True
            print_ok(f'Model "{MODEL}" found — local exploratory rerun is available.')
        else:
            print_warn(f'Model "{MODEL}" NOT found. Pull it first with: ollama pull {MODEL}')
    except Exception as e:
        print_err(f'Ollama not reachable: {e}')
        print_warn('Start Ollama with: ollama serve')
        print_warn(f'Then pull model : ollama pull {MODEL}')
else:
    print_info('Artifact-review mode requested explicitly. Fresh LLM calls are disabled.')

ARTIFACT_REVIEW = (not LLM_OK) and all(path.exists() for path in REPORT_FILES.values())
if ARTIFACT_REVIEW:
    print_ok('Saved baseline evaluation CSV outputs found — notebook will review artifacts without overwriting them.')
elif not LLM_OK:
    print_warn('No runnable backend detected and no saved baseline evaluation CSV outputs were found.')
    print_warn('Sections that depend on baseline outputs will stop early until one of those conditions is met.')

print_info(f'LLM_OK          = {LLM_OK}')
print_info(f'ARTIFACT_REVIEW = {ARTIFACT_REVIEW}')


## 2. Load OCEL Data and Benchmark

In [ ]:
print_banner('Section 2: Load Data', 'OCEL parquet + benchmark + schema catalog')

# ── DuckDB execution engine ────────────────────────────────────────────────
conn = duckdb.connect()
conn.execute(f"CREATE VIEW events    AS SELECT * FROM read_parquet('{OCEL_DIR}/events.parquet')")
conn.execute(f"CREATE VIEW objects   AS SELECT * FROM read_parquet('{OCEL_DIR}/objects.parquet')")
conn.execute(f"CREATE VIEW relations AS SELECT * FROM read_parquet('{OCEL_DIR}/relations.parquet')")

n_events    = conn.execute('SELECT COUNT(*) FROM events').fetchone()[0]
n_objects   = conn.execute('SELECT COUNT(*) FROM objects').fetchone()[0]
n_relations = conn.execute('SELECT COUNT(*) FROM relations').fetchone()[0]
print_ok(f'DuckDB views: events={n_events:,}  objects={n_objects:,}  relations={n_relations:,}')

# ── Benchmark ─────────────────────────────────────────────────────────────
bench_df = pd.read_csv(BENCHMARK)
if RUN_SUBSET:
    bench_df = bench_df[bench_df['qid'].isin(RUN_SUBSET)].reset_index(drop=True)
print_ok(f'Benchmark loaded: {len(bench_df)} questions')
print_info(f'Query classes: {bench_df["query_class"].value_counts().to_dict()}')
print_info(f'Difficulty    : {bench_df["difficulty"].value_counts().to_dict()}')

# ── Schema catalog ─────────────────────────────────────────────────────────
catalog   = json.loads(CATALOG.read_text(encoding='utf-8'))
whitelist = json.loads(WHITELIST.read_text(encoding='utf-8'))['whitelist']
allowed_joins = {w['relation_type'] for w in whitelist}
print_ok(f'Catalog loaded. Allowed join types: {sorted(allowed_joins)}')

## 3. Schema Text Builder

Used by B2 and B3: convert schema catalog to a compact text prompt.

In [ ]:
print_banner('Section 3: Schema Text Builder', 'Compact schema prompt for B2/B3')

def build_schema_text(catalog: dict, whitelist: list) -> str:
    """Compact schema text for prompt injection."""
    lines = [
        'DATABASE SCHEMA (DuckDB SQL):',
        '',
    ]
    for tbl in catalog['tables']:
        lines.append(f"Table: {tbl['table_name']}  ({tbl['row_count']:,} rows)")
        for col in tbl['columns']:
            null_note = f" [nullable, {col['null_pct']:.0f}% null]" if col['nullable'] else ''
            lines.append(f"  {col['name']}  {col['dtype']}{null_note}")
        if tbl['temporal_fields']:
            lines.append(f"  [temporal: {', '.join(tbl['temporal_fields'])}]")
        lines.append('')

    lines.append('EVENT TYPES (events.event_type):')
    for et in catalog['event_types']:
        lines.append(f"  '{et}'")
    lines.append('')

    lines.append('OBJECT TYPES (events.object_type / objects.object_type):')
    for ot in catalog['object_types']:
        lines.append(f"  '{ot}'")
    lines.append('')

    lines.append('ALLOWED JOINS (relations.relation_type only — do not hallucinate other joins):')
    for w in whitelist:
        lines.append(
            f"  '{w['relation_type']}'  "
            f"{w['from_object_type']} -> {w['to_object_type']}  "
            f"({w['n_links']:,} links)"
        )
    lines.append('')
    lines.append('NOTES:')
    lines.append('  - Use DuckDB SQL syntax.')
    lines.append('  - year(timestamp) and month(timestamp) are valid DuckDB functions.')
    lines.append('  - date_diff(\'day\', ts1, ts2) computes day difference.')
    lines.append('  - Do NOT use INSERT, UPDATE, DELETE, DROP, or CREATE.')
    return '\n'.join(lines)

SCHEMA_TEXT = build_schema_text(catalog, whitelist)
print_info(f'Schema text length: {len(SCHEMA_TEXT)} chars, {len(SCHEMA_TEXT.splitlines())} lines')
print_rule()
print(SCHEMA_TEXT[:600], '\n[... truncated ...]')

## 4. Few-Shot Examples Builder

Used by B3: select 3 solved examples from benchmark covering easy/medium/hard.

In [ ]:
print_banner('Section 4: Few-Shot Examples', 'Fixed 3-shot examples for exploratory few-shot prompt')

# Pick one easy, one medium, one hard from benchmark — fixed for reproducibility
# These are EXCLUDED from the evaluation set when computing B3 metrics
FEW_SHOT_QIDS = ['Q001', 'Q019', 'Q025']

few_shot_rows = bench_df[bench_df['qid'].isin(FEW_SHOT_QIDS)][['qid','nl_question','gold_sql']]

def build_fewshot_text(rows: pd.DataFrame) -> str:
    lines = ['SOLVED EXAMPLES:', '']
    for _, row in rows.iterrows():
        lines.append(f'Question: {row["nl_question"]}')
        lines.append(f'SQL: {row["gold_sql"]}')
        lines.append('')
    return '\n'.join(lines)

FEWSHOT_TEXT = build_fewshot_text(few_shot_rows)
print_ok(f'Few-shot examples (QIDs excluded from exploratory few-shot eval): {FEW_SHOT_QIDS}')
print_rule()
print(FEWSHOT_TEXT)

## 5. LLM Caller and SQL Extractor


In [ ]:
print_banner('Section 5: LLM Caller', 'DeepSeek / Ollama wrappers + SQL extractor')

DEEPSEEK_API_BASE = 'https://api.deepseek.com'

def call_ollama(prompt: str, model: str = MODEL, timeout: int = TIMEOUT_S) -> dict:
    payload = json.dumps({
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': TEMPERATURE,
            'num_predict': MAX_TOKENS,
        }
    }).encode('utf-8')

    t0 = time.time()
    try:
        req = urllib.request.Request(
            f'{OLLAMA_URL}/api/generate',
            data=payload,
            headers={'Content-Type': 'application/json'},
            method='POST',
        )
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = json.loads(resp.read())
        return {'text': data.get('response', ''), 'latency_s': round(time.time() - t0, 3), 'error': None}
    except Exception as e:
        return {'text': '', 'latency_s': round(time.time() - t0, 3), 'error': str(e)}

def call_deepseek(prompt: str, model: str = MODEL, timeout: int = TIMEOUT_S) -> dict:
    t0 = time.time()
    key = DEEPSEEK_API_KEY or ''
    if not key:
        return {'text': '', 'latency_s': 0.0, 'error': 'DEEPSEEK_API_KEY not set'}

    payload = json.dumps({
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': TEMPERATURE,
        'max_tokens': MAX_TOKENS,
    }).encode('utf-8')

    try:
        req = urllib.request.Request(
            f'{DEEPSEEK_API_BASE}/chat/completions',
            data=payload,
            headers={
                'Authorization': f'Bearer {key}',
                'Content-Type': 'application/json',
            },
            method='POST',
        )
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = json.loads(resp.read())
        text = data['choices'][0]['message']['content']
        return {'text': text, 'latency_s': round(time.time() - t0, 3), 'error': None}
    except Exception as e:
        return {'text': '', 'latency_s': round(time.time() - t0, 3), 'error': str(e)}

def call_llm(prompt: str) -> dict:
    if BACKEND == 'deepseek':
        return call_deepseek(prompt)
    if BACKEND == 'ollama':
        return call_ollama(prompt)
    return {'text': '', 'latency_s': 0.0, 'error': 'artifact review mode'}


In [ ]:
# SQL extraction + evaluation utilities
def extract_sql(llm_text: str) -> str:
    fenced = re.search(r'```(?:sql)?\s*([\s\S]+?)```', llm_text, re.IGNORECASE)
    if fenced:
        return fenced.group(1).strip()
    direct = re.search(r'(?is)(?:^|\n)\s*((?:select|with)\b[\s\S]+?)(?:;\s*$|\n\n|$)', llm_text)
    if direct:
        return direct.group(1).strip().rstrip(';')
    return llm_text.strip()

def result_hash(sql: str) -> tuple:
    try:
        df = conn.execute(sql).df()
        h = hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()
        return h, None
    except Exception as e:
        return None, str(e)

def check_join_hallucination(sql: str) -> bool:
    found = re.findall(r"relation_type\s*=\s*'([^']+)'", sql, re.IGNORECASE)
    for rt in found:
        if rt not in allowed_joins:
            return True
    return False

print_ok('LLM caller and SQL extractor ready.')
print_info(f'Allowed joins (for hallucination check): {sorted(allowed_joins)}')


## 6. Prompt Builders — B1, B2, B3

In [ ]:
print_banner('Section 6: Prompt Builders', 'One function per exploratory prompt variant')

SYSTEM_BASE = (
    'You are a SQL expert. Write a single DuckDB SQL query that answers the question.\n'
    'Return ONLY the SQL query, no explanation, no markdown unless wrapping SQL in ```sql blocks.\n'
    'Do NOT use INSERT, UPDATE, DELETE, DROP, or CREATE statements.\n'
)


def prompt_b1(question: str) -> str:
    """B1: Zero-shot — question only, no schema."""
    return (
        f'{SYSTEM_BASE}\n'
        f'The database contains SAP Order-to-Cash event log data in three tables: '
        f'events, objects, relations.\n\n'
        f'Question: {question}\n\n'
        f'SQL:'
    )


def prompt_b2(question: str) -> str:
    """B2: Schema-prompted — question + full schema text."""
    return (
        f'{SYSTEM_BASE}\n'
        f'{SCHEMA_TEXT}\n\n'
        f'Question: {question}\n\n'
        f'SQL:'
    )


def prompt_b3(question: str) -> str:
    """B3: Few-shot schema-prompted — question + schema + solved examples."""
    return (
        f'{SYSTEM_BASE}\n'
        f'{SCHEMA_TEXT}\n\n'
        f'{FEWSHOT_TEXT}\n'
        f'Question: {question}\n\n'
        f'SQL:'
    )

print_ok('Prompt builders ready.')

## 7. Exploratory Prompt Runner

Runs all three baselines over the benchmark. Skips B3 few-shot QIDs from its own eval.

In [ ]:
print_banner('Section 7: Exploratory Prompt Runner', 'B1 / B2 / B3 — all benchmark questions')

if LLM_OK:
    def run_baseline(method_name: str, prompt_fn, exclude_qids=None):
        results = []
        rows = bench_df[~bench_df['qid'].isin(exclude_qids or [])]
        total = len(rows)
        print_info(f'Running {method_name} on {total} questions...')

        for i, (_, row) in enumerate(rows.iterrows(), 1):
            qid       = row['qid']
            question  = row['nl_question']
            gold_hash = row['gold_result_hash']
            qclass    = row['query_class']
            diff      = row['difficulty']

            prompt   = prompt_fn(question)
            llm_out  = call_llm(prompt)
            raw_text = llm_out['text']
            latency  = llm_out['latency_s']
            llm_err  = llm_out['error']

            pred_sql = extract_sql(raw_text) if not llm_err else ''
            pred_hash, exec_err = result_hash(pred_sql) if pred_sql else (None, 'LLM error')

            exec_ok    = exec_err is None
            den_acc    = (pred_hash == gold_hash) if exec_ok else False
            join_hall  = check_join_hallucination(pred_sql) if pred_sql else False

            results.append({
                'method':      method_name,
                'qid':         qid,
                'query_class': qclass,
                'difficulty':  diff,
                'nl_question': question,
                'pred_sql':    pred_sql,
                'exec_ok':     exec_ok,
                'den_acc':     den_acc,
                'join_hall':   join_hall,
                'latency_s':   latency,
                'exec_error':  exec_err or '',
                'llm_error':   llm_err or '',
            })

            status = 'OK' if den_acc else ('EXEC_FAIL' if not exec_ok else 'WRONG')
            print(f'  [{i:02d}/{total}] {qid} ({qclass}/{diff}) -> {status}  ({latency:.1f}s)')

        return results

    print_rule()
    b1_results = run_baseline('B1_zero_shot', prompt_b1)
    print_rule()
    b2_results = run_baseline('baseline_schema_prompted', prompt_b2)
    print_rule()
    b3_results = run_baseline('baseline_few_shot_schema', prompt_b3, exclude_qids=FEW_SHOT_QIDS)
    print_rule()
    print_ok('All three baselines complete.')
elif ARTIFACT_REVIEW:
    b1_df = pd.read_csv(REPORT_FILES['B1'])
    b2_df = pd.read_csv(REPORT_FILES['B2'])
    b3_df = pd.read_csv(REPORT_FILES['B3'])
    b1_df['method'] = 'B1_zero_shot'
    b2_df['method'] = 'baseline_schema_prompted'
    b3_df['method'] = 'baseline_few_shot_schema'
    all_df = pd.concat([b1_df, b2_df, b3_df], ignore_index=True)
    print_ok('Loaded saved baseline evaluation baseline result CSVs for artifact review.')
    print_info(f'Rows loaded -> B1: {len(b1_df)}, B2: {len(b2_df)}, B3: {len(b3_df)}')
else:
    print_warn('No backend available and no saved artifacts to review. Section 7 stopped early.')


## 8. Save Raw Results

In [ ]:
print_banner('Section 8: Save Raw Results', 'CSV per baseline + combined parquet')

if 'all_df' not in globals() and LLM_OK:
    b1_df = pd.DataFrame(b1_results)
    b2_df = pd.DataFrame(b2_results)
    b3_df = pd.DataFrame(b3_results)
    all_df = pd.concat([b1_df, b2_df, b3_df], ignore_index=True)

if 'all_df' not in globals():
    print_warn('Skipped — no baseline result tables are currently in memory.')
elif LLM_OK:
    b1_df.to_csv(REPORTS_DIR / 'nb03_b1_results.csv', index=False)
    b2_df.to_csv(REPORTS_DIR / 'nb03_b2_results.csv', index=False)
    b3_df.to_csv(REPORTS_DIR / 'nb03_b3_results.csv', index=False)
    all_df.to_parquet(REPORTS_DIR / 'nb03_all_results.parquet', index=False)

    print_ok(f'Saved: nb03_b1_results.csv ({len(b1_df)} rows)')
    print_ok(f'Saved: nb03_b2_results.csv ({len(b2_df)} rows)')
    print_ok(f'Saved: nb03_b3_results.csv ({len(b3_df)} rows)')
    print_ok(f'Saved: nb03_all_results.parquet ({len(all_df)} rows total)')
else:
    print_info('Artifact-review mode — using saved baseline evaluation CSVs without overwriting them.')


## 9. Metrics Computation

Primary: ExecRate, DenAcc, HallRate, JoinHallRate  
Secondary: breakdown by query_class and difficulty

In [ ]:
print_banner('Section 9: Metrics', 'ExecRate / DenAcc / HallRate / JoinHallRate')

if 'all_df' not in globals():
    print_warn('Skipped — baseline outputs are not available in memory.')
else:
    def compute_metrics(df: pd.DataFrame, method: str) -> dict:
        n = len(df)
        exec_rate   = df['exec_ok'].mean()
        den_acc     = df['den_acc'].mean()
        hall_rate   = 1 - exec_rate
        join_hall   = df['join_hall'].mean()
        avg_latency = df['latency_s'].mean()
        return {
            'method':        method,
            'n_questions':   n,
            'ExecRate':      round(exec_rate,   4),
            'DenAcc':        round(den_acc,     4),
            'HallRate':      round(hall_rate,   4),
            'JoinHallRate':  round(join_hall,   4),
            'AvgLatency_s':  round(avg_latency, 3),
        }

    metrics = [
        compute_metrics(b1_df, 'B1_zero_shot'),
        compute_metrics(b2_df, 'baseline_schema_prompted'),
        compute_metrics(b3_df, 'baseline_few_shot_schema'),
    ]
    metrics_df = pd.DataFrame(metrics)

    print_rule()
    print('OVERALL METRICS:')
    print_rule()
    try:
        styled_df(metrics_df)
    except Exception:
        print(metrics_df.to_string(index=False))

    print_rule()
    print('DenAcc BY QUERY CLASS:')
    class_breakdown = all_df.groupby(['method','query_class'])['den_acc'].mean().unstack(fill_value=0).round(3)
    try:
        styled_df(class_breakdown.reset_index())
    except Exception:
        print(class_breakdown.to_string())

    print_rule()
    print('DenAcc BY DIFFICULTY:')
    diff_breakdown = all_df.groupby(['method','difficulty'])['den_acc'].mean().unstack(fill_value=0).round(3)
    try:
        styled_df(diff_breakdown.reset_index())
    except Exception:
        print(diff_breakdown.to_string())

    metrics_out = {
        'overall':        metrics,
        'by_query_class': class_breakdown.to_dict(),
        'by_difficulty':  diff_breakdown.to_dict(),
    }
    (REPORTS_DIR / 'nb03_baseline_metrics.json').write_text(json.dumps(metrics_out, indent=2), encoding='utf-8')
    print_ok('Metrics saved: outputs/reports/nb03_baseline_metrics.json')


## 10. Error Analysis — Join Hallucination Detail

In [ ]:
print_banner('Section 11: Error Analysis', 'Join hallucinations + exec failure taxonomy')

if 'all_df' not in globals():
    print_warn('Skipped — baseline outputs are not available in memory.')
else:
    print_rule()
    print('JOIN HALLUCINATION CASES (pred_sql uses relation_type not in whitelist):')
    hall_cases = all_df[all_df['join_hall'] == True][['method','qid','query_class','difficulty','pred_sql']]
    if len(hall_cases) == 0:
        print_ok('No join hallucinations detected.')
    else:
        print_warn(f'{len(hall_cases)} join hallucination cases found:')
        try:
            styled_df(hall_cases[['method','qid','query_class','difficulty']])
        except Exception:
            print(hall_cases[['method','qid','query_class','difficulty']].to_string(index=False))

    print_rule()
    print('EXECUTION FAILURE CASES (SQL did not execute):')
    fail_cases = all_df[~all_df['exec_ok'].astype(bool)][['method','qid','query_class','difficulty','exec_error']]
    if len(fail_cases) == 0:
        print_ok('All queries executed successfully.')
    else:
        print_warn(f'{len(fail_cases)} execution failures:')
        try:
            styled_df(fail_cases[['method','qid','query_class','difficulty','exec_error']].head(20))
        except Exception:
            print(fail_cases[['method','qid','query_class','difficulty','exec_error']].head(20).to_string(index=False))

    print_rule()
    print('WRONG RESULT CASES (executed but result hash mismatch):')
    wrong_cases = all_df[all_df['exec_ok'].astype(bool) & ~all_df['den_acc'].astype(bool)]
    print_info(f'Total wrong (exec OK, wrong result): {len(wrong_cases)}')
    by_class = wrong_cases.groupby(['method','query_class']).size().unstack(fill_value=0)
    try:
        styled_df(by_class.reset_index())
    except Exception:
        print(by_class.to_string())


## 11. Key Findings Summary

In [ ]:
print_banner('Section 12: Key Findings', 'baseline evaluation summary — baseline reference point established')

if 'all_df' not in globals():
    print_warn('Cannot compute findings — baseline outputs are not available in memory.')
else:
    findings = []
    b1m, b2m, b3m = metrics[0], metrics[1], metrics[2]

    findings.append({
        'Finding': 'B1 -> B2 DenAcc delta',
        'Value': f"+{(b2m['DenAcc']-b1m['DenAcc'])*100:.1f}pp",
        'Meaning': 'Schema context effect on accuracy (no schema -> full schema)'
    })
    findings.append({
        'Finding': 'B2 -> B3 DenAcc delta',
        'Value': f"+{(b3m['DenAcc']-b2m['DenAcc'])*100:.1f}pp",
        'Meaning': 'Few-shot examples effect on accuracy'
    })
    findings.append({
        'Finding': 'Best baseline DenAcc',
        'Value': f"{max(b1m['DenAcc'],b2m['DenAcc'],b3m['DenAcc'])*100:.1f}%",
        'Meaning': 'Ceiling for unconstrained methods — constrained pipeline must beat this'
    })
    findings.append({
        'Finding': 'B1 JoinHallRate',
        'Value': f"{b1m['JoinHallRate']*100:.1f}%",
        'Meaning': 'Fraction of B1 queries hallucinating invalid relation_types'
    })
    findings.append({
        'Finding': 'B2 JoinHallRate',
        'Value': f"{b2m['JoinHallRate']*100:.1f}%",
        'Meaning': 'Fraction of B2 queries hallucinating invalid relation_types (schema-prompted)'
    })

    findings_df = pd.DataFrame(findings)
    try:
        styled_df(findings_df)
    except Exception:
        print(findings_df.to_string(index=False))

    print_rule()
    print_gate('baseline evaluation: Baseline translators implemented and evaluated.', True)
    if ARTIFACT_REVIEW:
        print_info('This notebook reviewed the canonical saved baseline evaluation outputs without rerunning LLM calls.')
    elif BACKEND == 'deepseek':
        print_info('This notebook reviews the reported DeepSeek baseline evaluation outputs for the current dev split.')
    else:
        print_info('This notebook used the local Ollama fallback for interactive exploration.')
    print_ok('Next: constrained pipeline — constrained pipeline (IR + verifier + join whitelist enforcement).')
    print_info('The metrics above become the reference row in the final ablation table.')


## References

1. Gao et al. (2023). *Text-to-SQL Empowered by Large Language Models: A Benchmark Evaluation*. [NQ016]
2. Pourreza & Rafiei (2024). *DIN-SQL: Decomposed In-Context Learning of Text-to-SQL with Self-Correction*. [NQ017]  
   — B2 schema-prompted design follows the DIN-SQL schema serialisation pattern.
3. Rajkumar et al. (2022). *Evaluating the Text-to-SQL Capabilities of Large Language Models*. [NQ018]
4. Sun et al. (2023). *SQL-PaLM: Improved Large Language Model Adaptation for Text-to-SQL*. [NQ031]
5. van der Aalst (2022). *Object-Centric Process Mining*. [NQ066]  
   — OCEL schema and relation graph motivation.

---

# Part 2 — Constrained Pipeline (Method M) Artifact Review

Demonstrates the four pipeline stages using saved eval results from `outputs/reports/nb04_method_m_dev.csv`.

**Prerequisite:** Part 1 (baseline runner) must have written `nb03_b*.csv` result files.  
All comparison figures are generated in `03_phase2_results.ipynb`.


In [ ]:
# ROOT and imports already in scope from Part 1 setup cell above.
import pandas as pd
from pathlib import Path
import json, warnings
warnings.filterwarnings("ignore")

REPORT = ROOT / "outputs" / "reports" / "nb04_method_m_dev.csv"
mm = pd.read_csv(REPORT)
print(f"Loaded {len(mm)} dev results from {REPORT.name}")
print(f"Statuses: {mm['status'].value_counts().to_dict()}")
print(f"Mean DenAcc: {mm['den_acc'].mean():.1%}")


## 1. Count filter - pipeline succeeds (Q004)

**Question:** "How many billing_created events are there?"  
**Class:** `count_filter` | **Difficulty:** easy | **Status:** accept (1st attempt)

This shows the pipeline in its simplest successful path: schema retrieval narrows to the events table, LLM emits a typed IR with `event_type = billing_created`, verifier accepts on first try, compiler emits a single-table SELECT.


In [ ]:
q4 = mm[mm["qid"] == "Q004"].iloc[0]

print("─" * 60)
print(f"Question : {q4.nl_question}")
print(f"Class    : {q4.query_class}  Difficulty: {q4.difficulty}")
print()
print("Stage 1 - Schema retrieval (TF-IDF top-k slice):")
print("  Tables retrieved: events  ->  columns: event_id, event_type, object_id, object_type, timestamp")
print()
print("Stage 2 - IR (LLM output, deepseek-chat):")
print("  intent      : count_filter")
print("  filters     : [{col: event_type, op: =, val: billing_created}]")
print("  aggregation : count")
print()
print("Stage 3 - Verifier:")
print("  event_type 'billing_created' in enum_values  OK")
print("  no join required for count_filter  OK")
print(f"  Decision: ACCEPT  (ir_attempts={q4.ir_attempts})")
print()
print("Stage 4 - Compiled SQL:")
print(q4.pred_sql)
print()
print(f"Result: exec_ok={q4.exec_ok}  den_acc={q4.den_acc}  latency={q4.latency_s:.1f}s")


## 2. Anomaly filter - 3-CTE compiler succeeds (Q028)

**Question:** "How many billing-to-payment cases take more than 873 days?"  
**Class:** `anomaly_filter` | **Difficulty:** hard | **Status:** accept (1st attempt)

This shows the specialised 3-CTE compiler path: two event anchors (billing_created, payment_clearing) joined via the `billing_to_ar` relation with a `date_diff` threshold. The verifier checks both the numeric threshold and that the relation type is whitelisted.


In [ ]:
q28 = mm[mm["qid"] == "Q028"].iloc[0]

print("─" * 60)
print(f"Question : {q28.nl_question}")
print(f"Class    : {q28.query_class}  Difficulty: {q28.difficulty}")
print()
print("Stage 1 - Schema retrieval:")
print("  Tables: events, relations  ->  billing_to_ar relation type identified")
print()
print("Stage 2 - IR:")
print("  intent      : anomaly_filter")
print("  filters     : [{col: delay_days, op: >, val: 873}]")
print("  joins       : [{relation_type: billing_to_ar}]")
print("  event anchors: billing_created -> payment_clearing")
print()
print("Stage 3 - Verifier:")
print("  delay_days in VIRTUAL_COLS (computed column, exempt from schema check)  OK")
print("  billing_to_ar in relation whitelist  OK")
print("  numeric threshold present  OK")
print(f"  Decision: ACCEPT  (ir_attempts={q28.ir_attempts})")
print()
print("Stage 4 - Compiled SQL (3-CTE pattern):")
print(q28.pred_sql)
print()
print(f"Result: exec_ok={q28.exec_ok}  den_acc={q28.den_acc}  latency={q28.latency_s:.1f}s")


## 3. Anomaly filter - semantic failure after 3 attempts (Q029)

**Question:** "Count long clearing gaps above 873 days."  
**Class:** `anomaly_filter` | **Difficulty:** hard | **Status:** accept (3 attempts) | **DenAcc: 0**

This shows a genuine semantic failure. The question asks for *clearing* gaps (billing -> payment = `billing_to_ar`), but after 3 attempts the LLM chose `order_to_billing` - a plausible but wrong relation type. The verifier accepted the IR because `order_to_billing` *is* whitelisted; it cannot infer the semantic intent from "clearing gaps." This is the primary failure mode for anomaly_filter class.


In [ ]:
q29 = mm[mm["qid"] == "Q029"].iloc[0]

print("─" * 60)
print(f"Question : {q29.nl_question}")
print(f"Class    : {q29.query_class}  Difficulty: {q29.difficulty}")
print()
print("Stage 2 - IR (attempt 3):")
print("  intent      : anomaly_filter")
print("  filters     : [{col: delay_days, op: >, val: 873}]")
print("  joins       : [{relation_type: order_to_billing}]   ← WRONG (should be billing_to_ar)")
print()
print("Stage 3 - Verifier:")
print("  delay_days in VIRTUAL_COLS  OK")
print("  order_to_billing in whitelist  OK  (verifier cannot detect semantic mismatch)")
print("  numeric threshold present  OK")
print(f"  Decision: ACCEPT  (ir_attempts={q29.ir_attempts})")
print()
print("Stage 4 - Compiled SQL:")
print(q29.pred_sql)
print()
print(f"Result: exec_ok={q29.exec_ok}  den_acc={q29.den_acc}  latency={q29.latency_s:.1f}s")
print()
print("Root cause: 'clearing gaps' semantically implies payment clearing (billing_to_ar).")
print("The verifier enforces structural validity but not semantic intent matching.")
print("Repair update: add relation-type hint examples keyed to 'clearing' vocabulary.")


In [ ]:
from nl2ocel.query_verifier import verify_ir, load_schema_index, load_whitelist_set, load_sql_policy

schema_index  = load_schema_index(ROOT / "configs" / "schema_catalog.json")
allowed_joins = load_whitelist_set(ROOT / "configs" / "relation_whitelist.json")
policy        = load_sql_policy(ROOT / "configs" / "sql_policy.yaml")

# Hallucinated IR: directly joins order_item â†' ar_item (not in whitelist)
bad_ir = {
    "intent":   "count_filter",
    "tables":   ["events", "relations"],
    "select":   [{"col": "event_id", "agg": "COUNT", "alias": "n"}],
    "filters":  [],
    "joins":    [{"relation_type": "order_to_ar", "from_alias": "o", "to_alias": "a",
                  "join_col": "from_object_id"}],
    "group_by": [], "order_by": [], "limit": None, "temporal": None, "ctes": [],
}

result = verify_ir(bad_ir, schema_index, allowed_joins, policy)
final_status = "reject" if result.status in {"reject", "repair"} else result.status
print(f"Verifier final status: {final_status}")
print(f"Verifier signal: {result.status}")
print(f"Errors:          {result.errors}")
print(f"Repair hint:     {json.dumps(result.repair_hint, indent=2)}")
print()
print("'order_to_ar' is NOT in the whitelist - the verifier blocks this IR before execution.")
print("A baseline (B1) would compile and execute this, returning silently wrong results.")


## 4. Summary

| Query | Class | Difficulty | Attempts | DenAcc | Outcome |
|-------|-------|------------|----------|--------|---------|
| Q004 - billing_created count | count_filter | easy | 1 | OK | Verifier accepts, single-table SQL |
| Q028 - billing-to-payment >873d | anomaly_filter | hard | 1 | OK | 3-CTE compiler, threshold enforced |
| Q029 - clearing gaps >873d | anomaly_filter | hard | 3 | ✗ | Verifier accepts wrong relation type |

**Key finding:** The verifier eliminates structurally invalid IRs (hallucinated columns, non-whitelisted joins) but cannot detect *semantic* mismatches between synonymous relation types. This is the primary residual failure mode. See `05_phase1_eval.ipynb` for full quantitative results.
